In [1]:
from pathlib import Path
import json
import time

import pandas as pd
import torch

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd().parent

EXPERIMENTS_DIR = (
    PROJECT_ROOT / "experiments"
)

MODELS_DIR = (
    PROJECT_ROOT / "models"
)

EXPORT_DIR = (
    EXPERIMENTS_DIR
    / "exports"
)

EXPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)

print("Inspectra — Model Export")
print("=" * 60)
print("Device:", DEVICE)
print("Export directory:", EXPORT_DIR)

Inspectra — Model Export
Device: 0
Export directory: d:\Inspectra\experiments\exports


In [2]:
def find_model(
    dataset_name
):

    candidates = list(
        EXPERIMENTS_DIR.rglob(
            f"{dataset_name}*/**/best.pt"
        )
    )

    if not candidates:
        return None

    candidates = [
        path
        for path in candidates
        if "finetun" in str(path).lower()
    ]

    if candidates:
        return candidates[0]

    return None


bottle_model_path = find_model(
    "bottle"
)

pcb_model_path = find_model(
    "pcb"
)

print(
    "Bottle:",
    bottle_model_path
)

print(
    "PCB:",
    pcb_model_path
)

Bottle: d:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.pt
PCB: d:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.pt


In [3]:
BOTTLE_MODEL = bottle_model_path
PCB_MODEL = pcb_model_path

In [4]:
def export_yolo_onnx(
    dataset_name,
    model_path
):

    if model_path is None:
        print(
            f"{dataset_name}: model not found"
        )
        return None

    if not model_path.exists():
        print(
            f"{dataset_name}: "
            f"model does not exist"
        )
        return None

    model = YOLO(
        str(model_path)
    )

    output_dir = (
        EXPORT_DIR
        / dataset_name
    )

    output_dir.mkdir(
        parents=True,
        exist_ok=True
    )

    start = time.perf_counter()

    exported = model.export(
        format="onnx",
        imgsz=640,
        opset=17,
        simplify=True,
        dynamic=False,
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    exported_path = Path(
        exported
    )

    destination = (
        output_dir
        / f"{dataset_name}_finetuned.onnx"
    )

    if exported_path.resolve() != destination.resolve():

        destination.write_bytes(
            exported_path.read_bytes()
        )

    size_mb = (
        destination.stat().st_size
        / (1024 ** 2)
    )

    result = {
        "dataset": dataset_name,
        "format": "ONNX",
        "source": str(model_path),
        "output": str(destination),
        "size_mb": size_mb,
        "export_seconds": elapsed,
    }

    print(
        f"\n{dataset_name.upper()}"
    )

    print(
        "Output:",
        destination
    )

    print(
        f"Size: {size_mb:.2f} MB"
    )

    print(
        f"Export time: {elapsed:.2f}s"
    )

    return result

In [5]:
bottle_export = export_yolo_onnx(
    "bottle",
    BOTTLE_MODEL
)

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13450HX)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'd:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 8, 8400) (6.0 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success  2.3s, saved as 'd:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.onnx' (11.7 MB)

Export complete (2.6s)
Results saved to D:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.onnx
Predict:         yolo predict task=detect model=d:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=d:\Inspectra\experiments\finetuning\bottle\finetuned\weights\best.onnx imgsz=640 data=d:\Inspectra\datasets\processed\bottle\data.yaml  
Visualize:       htt

In [6]:
pcb_export = export_yolo_onnx(
    "pcb",
    PCB_MODEL
)

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CPU (13th Gen Intel Core i5-13450HX)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from 'd:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 10, 8400) (6.0 MB)

ONNX: starting export with onnx 1.22.0 opset 17...
ONNX: slimming with onnxslim 0.1.95...
ONNX: export success  1.0s, saved as 'd:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.onnx' (11.7 MB)

Export complete (1.2s)
Results saved to D:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.onnx
Predict:         yolo predict task=detect model=d:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.onnx imgsz=640 
Validate:        yolo val task=detect model=d:\Inspectra\experiments\finetuning\pcb\finetuned\weights\best.onnx imgsz=640 data=d:\Inspectra\datasets\processed\pcb\data.yaml  
Visualize:       https://netron.app



In [7]:
from torchvision.models import resnet18

ROAD_MODEL = (
    MODELS_DIR
    / "road"
    / "finetuned"
    / "resnet18_finetuned.pt"
)

road_device = (
    f"cuda:{DEVICE}"
    if isinstance(DEVICE, int)
    else DEVICE
)

road_model = resnet18(
    weights=None
)

road_model.fc = torch.nn.Linear(
    road_model.fc.in_features,
    2
)

road_state = torch.load(
    ROAD_MODEL,
    map_location="cpu"
)

road_model.load_state_dict(
    road_state
)

road_model.eval()

print(
    "Road model loaded:"
)

print(
    ROAD_MODEL
)

Road model loaded:
d:\Inspectra\models\road\finetuned\resnet18_finetuned.pt


C:\Users\Garvit\AppData\Local\Temp\ipykernel_1308\4291773012.py:25: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  road_state = torch.load(


In [8]:
road_export_dir = (
    EXPORT_DIR / "road"
)

road_export_dir.mkdir(
    parents=True,
    exist_ok=True
)

road_onnx_path = (
    road_export_dir
    / "road_resnet18_finetuned.onnx"
)

dummy_input = torch.randn(
    1,
    3,
    224,
    224
)

start = time.perf_counter()

torch.onnx.export(
    road_model,
    dummy_input,
    road_onnx_path,
    input_names=[
        "images"
    ],
    output_names=[
        "logits"
    ],
    dynamic_axes={
        "images": {
            0: "batch"
        },
        "logits": {
            0: "batch"
        },
    },
    opset_version=17,
    do_constant_folding=True,
)

elapsed = (
    time.perf_counter()
    - start
)

road_size_mb = (
    road_onnx_path.stat().st_size
    / (1024 ** 2)
)

print(
    "Output:",
    road_onnx_path
)

print(
    f"Size: {road_size_mb:.2f} MB"
)

print(
    f"Export time: {elapsed:.2f}s"
)

Output: d:\Inspectra\experiments\exports\road\road_resnet18_finetuned.onnx
Size: 42.63 MB
Export time: 0.44s


In [9]:
import onnx
def validate_onnx(
    path
):

    print(
        f"\nValidating:"
        f" {path}"
    )

    model = onnx.load(
        str(path)
    )

    onnx.checker.check_model(
        model
    )

    print(
        "ONNX validation: PASS"
    )

    print(
        "IR version:",
        model.ir_version
    )

    print(
        "Opset:",
        model.opset_import[0].version
    )

    print(
        "Inputs:",
        [
            item.name
            for item in model.graph.input
        ]
    )

    print(
        "Outputs:",
        [
            item.name
            for item in model.graph.output
        ]
    )

    return True

In [10]:
onnx_files = list(
    EXPORT_DIR.rglob(
        "*.onnx"
    )
)

for file in onnx_files:

    validate_onnx(
        file
    )


Validating: d:\Inspectra\experiments\exports\bottle\bottle_finetuned.onnx
ONNX validation: PASS
IR version: 8
Opset: 17
Inputs: ['images']
Outputs: ['output0']

Validating: d:\Inspectra\experiments\exports\pcb\pcb_finetuned.onnx
ONNX validation: PASS
IR version: 8
Opset: 17
Inputs: ['images']
Outputs: ['output0']

Validating: d:\Inspectra\experiments\exports\road\road_resnet18_finetuned.onnx
ONNX validation: PASS
IR version: 8
Opset: 17
Inputs: ['images']
Outputs: ['logits']


In [11]:
export_results = []

for result in [
    bottle_export,
    pcb_export,
]:

    if result is not None:

        export_results.append(
            result
        )

export_results.append(
    {
        "dataset": "road",
        "format": "ONNX",
        "source": str(
            ROAD_MODEL
        ),
        "output": str(
            road_onnx_path
        ),
        "size_mb": road_size_mb,
        "export_seconds": elapsed,
    }
)

export_df = pd.DataFrame(
    export_results
)

display(
    export_df
)

,dataset,format,source,output,size_mb,export_seconds
0,bottle,ONNX,d:\Inspectra\experiments\finetuning\bottle\fin...,d:\Inspectra\experiments\exports\bottle\bottle...,11.699328,2.571063
1,pcb,ONNX,d:\Inspectra\experiments\finetuning\pcb\finetu...,d:\Inspectra\experiments\exports\pcb\pcb_finet...,11.700858,1.245439
2,road,ONNX,d:\Inspectra\models\road\finetuned\resnet18_fi...,d:\Inspectra\experiments\exports\road\road_res...,42.629870,0.440273


In [12]:
manifest_path = (
    EXPORT_DIR
    / "export_manifest.json"
)

with open(
    manifest_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        export_results,
        file,
        indent=4
    )

export_df.to_csv(
    EXPORT_DIR
    / "export_manifest.csv",
    index=False
)

print(
    "Manifest:",
    manifest_path
)

Manifest: d:\Inspectra\experiments\exports\export_manifest.json


In [13]:
import onnxruntime as ort

print(
    "ONNX Runtime:",
    ort.__version__
)

print(
    "Providers:"
)

for provider in (
    ort.get_available_providers()
):

    print(
        " -",
        provider
    )

ONNX Runtime: 1.28.0
Providers:
 - AzureExecutionProvider
 - CPUExecutionProvider


In [14]:
import numpy as np
from PIL import Image


def load_image_tensor(
    image_path,
    size=640
):

    image = Image.open(
        image_path
    ).convert("RGB")

    image = image.resize(
        (size, size)
    )

    image = np.asarray(
        image,
        dtype=np.float32
    )

    image = (
        image / 255.0
    )

    image = np.transpose(
        image,
        (2, 0, 1)
    )

    image = np.expand_dims(
        image,
        axis=0
    )

    return image

In [15]:
def test_onnx_session(
    model_path,
    image_path
):

    session = ort.InferenceSession(
        str(model_path),
        providers=[
            "CPUExecutionProvider"
        ]
    )

    input_name = (
        session.get_inputs()[0].name
    )

    image = load_image_tensor(
        image_path
    )

    outputs = session.run(
        None,
        {
            input_name: image
        }
    )

    print(
        "Model:",
        model_path
    )

    print(
        "Input:",
        input_name
    )

    for index, output in enumerate(
        outputs
    ):

        print(
            f"Output {index}:",
            output.shape
        )

    return outputs